# Query Understanding for Agent Routing: Few-Shot + Reasoned Classification

Multi-node agents live or die by their first node: **intent classification**. If "sort these by severity" gets routed to the search node, everything downstream is wasted compute.

This notebook shows the pattern used in a production 5-node agent (intent → search → categorize → sort → recommend):

1. Structured-output classification over a **closed intent set**
2. Few-shot examples + a required `reason` field (asking the model to justify its choice measurably improves consistency)
3. A tiny labeled set to **measure** zero-shot vs. few-shot before shipping

## 1. Setup

In [ ]:
%pip install openai pandas -q

import json
import pandas as pd
from openai import OpenAI

client = OpenAI()
MODEL = "gpt-4o-mini"

INTENTS = ["search_issues", "categorize", "sort", "recommend_resolution"]

# Small labeled evaluation set (in production: curated from real user queries)
eval_set = [
    ("Show me past quality issues for the TAC950",              "search_issues"),
    ("Any leak problems reported on heat pumps?",               "search_issues"),
    ("Group those issues by root cause",                        "categorize"),
    ("Break the results down by product class",                 "categorize"),
    ("Order them by severity, worst first",                     "sort"),
    ("Which of these happened most recently?",                  "sort"),
    ("How was the valve leak fixed last time?",                 "recommend_resolution"),
    ("What should we do about this ignition failure?",          "recommend_resolution"),
]

## 2. Zero-shot vs. few-shot + reason

Both use strict JSON output. The few-shot variant adds intent-query example pairs and requires a one-sentence `reason` — a lightweight chain-of-thought that stabilizes borderline cases (e.g., "which happened most recently?" is a *sort*, not a *search*).

In [ ]:
def classify(query: str, few_shot: bool) -> str:
    system = f"""Classify the user's query into exactly one intent: {INTENTS}.
Respond in JSON: {{"reason": "<one sentence>", "intent": "<intent>"}}"""
    if few_shot:
        system += """

Examples:
- "find similar leak cases" -> search_issues (asks to retrieve past issues)
- "bundle them by cause type" -> categorize (asks to group an existing list)
- "most severe first please" -> sort (asks to reorder an existing list)
- "how do we fix it" -> recommend_resolution (asks for a countermeasure)"""
    resp = client.chat.completions.create(
        model=MODEL,
        response_format={"type": "json_object"},
        messages=[{"role": "system", "content": system},
                  {"role": "user", "content": query}],
        temperature=0,
    )
    return json.loads(resp.choices[0].message.content)["intent"]

rows = []
for mode, fs in [("zero-shot", False), ("few-shot + reason", True)]:
    correct = sum(classify(q, fs) == y for q, y in eval_set)
    rows.append((mode, f"{correct}/{len(eval_set)}"))

pd.DataFrame(rows, columns=["prompt strategy", "accuracy"])

## 3. Routing

With a reliable classifier, routing is a dictionary lookup — keep it deterministic.

In [ ]:
HANDLERS = {
    "search_issues":         lambda q: f"[search node] querying index for: {q}",
    "categorize":            lambda q: f"[categorize node] grouping current results",
    "sort":                  lambda q: f"[sort node] reordering current results",
    "recommend_resolution":  lambda q: f"[resolution node] looking up fix history",
}

q = "Order them by severity, worst first"
print(HANDLERS[classify(q, few_shot=True)](q))

## Takeaways

- **Closed intent set + structured output** turns a fuzzy LLM call into a testable component.
- **Require a reason** — a one-sentence justification field is cheap and measurably stabilizes classification.
- **Measure before shipping** — even 8 labeled queries catch routing regressions; in production, keep growing this set from real traffic and re-run it whenever the prompt changes (prompts are code: version them).

*Pattern from a production 5-node agent. Full write-up: [case study](../case-study-2-rag-search-system.md)*